# Driver Drowsiness - threshold fitting (Google Colab)

This notebook fits the detector's thresholds to YOUR data and exports a
`learned_thresholds.json` that the live system picks up automatically.

Pipeline: mount Drive -> extract per-frame features (same maths as runtime)
-> look at each class's distributions -> pick the thresholds that best
separate the classes -> save the JSON back to the project folder.

**Expected Drive layout** (put both in your Google Drive):
```
MyDrive/
  alertVSdrowsy/        <- this whole project folder (so we can import it)
  dms_data/             <- your labelled clips
    Yawning/  *.mp4
    Singing/  *.mp4
    Alert/    *.mp4
    Drowsy/   *.mp4
    Sleeping/ *.mp4
    Distracted/ *.mp4
```

In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Install the deps the extractor needs (Colab has no MediaPipe by default).
#    We do NOT need TensorFlow; the extractor reuses the project's video model.
!pip -q install mediapipe opencv-python numpy

In [ ]:
# 3) Point at your Drive folders and run feature extraction.
import os
PROJECT = '/content/drive/MyDrive/alertVSdrowsy'   # the project folder in Drive
DATA_DIR = '/content/drive/MyDrive/dms_data'        # your labelled clips
OUT_CSV  = '/content/features.csv'

# Run the SAME extractor the repo ships (train/runtime parity).
!cd "$PROJECT/train" && python extract_features.py --videos "$DATA_DIR" --out "$OUT_CSV"
print('features ->', OUT_CSV)

In [ ]:
# 4) Load the features and look at each class. Only frames with a detected face
#    matter for fitting thresholds.
#    NOTE: if you ran annotate_yawns.py, a 'Yawning' clip now contributes 'Yawning'
#    rows only for the marked yawn span; its closed-mouth frames appear as 'Neutral'
#    and any IGNORE frames were dropped by the extractor. So expect a 'Neutral' class
#    here and a smaller, cleaner 'Yawning' class.
import pandas as pd
df = pd.read_csv(OUT_CSV)
df = df[df.face_found == 1].copy()
print('frames per class:')
print(df.label.value_counts())
df.groupby('label')[['ear','mar','lip_gap_level']].describe().T

In [ ]:
# 5) A tiny helper: pick the threshold that best separates two groups.
#    'greater' means the positive class has LARGER values (e.g. MAR for yawning);
#    'less' means SMALLER (e.g. EAR for closed eyes). We scan candidate cut
#    points and keep the one with the best balanced accuracy (Youden's J).
import numpy as np

def best_threshold(pos, neg, direction='greater'):
    pos, neg = np.asarray(pos), np.asarray(neg)
    cands = np.unique(np.concatenate([pos, neg]))
    best_t, best_j = None, -1
    for t in cands:
        if direction == 'greater':
            tpr = (pos >= t).mean(); fpr = (neg >= t).mean()
        else:
            tpr = (pos <= t).mean(); fpr = (neg <= t).mean()
        j = tpr - fpr
        if j > best_j:
            best_j, best_t = j, float(t)
    return best_t, best_j

In [ ]:
# 6) Fit the three core thresholds from the data.
def vals(labels, col):
    return df[df.label.isin(labels)][col].values

# 'Neutral' is produced by annotate_yawns.py: the closed-mouth frames OUTSIDE the
# marked yawn span in each Yawning clip. They are now training NEGATIVES (they used
# to be mislabelled 'Yawning' and dragged these cuts DOWN), so we add them to the
# negative pools below. (No 'Neutral' rows exist if you haven't annotated -> no-op.)

# MAR_YAWN: yawning mouth is wide open vs everything else.
mar_t, mar_j = best_threshold(vals(['Yawning'], 'mar'),
                              vals(['Singing','Alert','Distracted','Neutral'], 'mar'),
                              'greater')

# MMS_YAWN_LEVEL: yawning lip-gap level vs talking/singing/closed-mouth.
lvl_t, lvl_j = best_threshold(vals(['Yawning'], 'lip_gap_level'),
                              vals(['Singing','Alert','Neutral'], 'lip_gap_level'),
                              'greater')

# EAR_CLOSED: eyes-closed states (Sleeping/Drowsy) vs eyes-open (Alert).
ear_t, ear_j = best_threshold(vals(['Sleeping','Drowsy'], 'ear'),
                              vals(['Alert'], 'ear'),
                              'less')

print(f'MAR_YAWN       = {mar_t:.3f}  (separation J={mar_j:.2f})')
print(f'MMS_YAWN_LEVEL = {lvl_t:.3f}  (separation J={lvl_j:.2f})')
print(f'EAR_CLOSED     = {ear_t:.3f}  (separation J={ear_j:.2f})')

In [ ]:
# 7) Save the fitted thresholds next to config.py. The live system loads this
#    file at startup and overrides its defaults - no code change needed.
#    (Only include a key if its separation J is decent, so a noisy fit can't
#     make the live system worse than the calibrated defaults.)
import json
learned = {}
if mar_j > 0.5: learned['MAR_YAWN'] = round(mar_t, 3)
if lvl_j > 0.5: learned['MMS_YAWN_LEVEL'] = round(lvl_t, 3)
if ear_j > 0.5: learned['EAR_CLOSED'] = round(ear_t, 3)

dest = os.path.join(PROJECT, 'learned_thresholds.json')
with open(dest, 'w') as f:
    json.dump(learned, f, indent=2)
print('wrote', dest)
print(json.dumps(learned, indent=2))

## Done

`learned_thresholds.json` is now in the project folder. Next time you run
`python main.py`, `config.py` loads it automatically and prints which thresholds
it applied. Delete the file to go back to the shipped defaults.

**Where to go further** (optional): instead of single thresholds you could fit a
small classifier (e.g. logistic regression / a tiny LSTM) on these same feature
rows. The parent project's `train.py` / `cross_validate.py` already do exactly
that on this data with subject-grouped cross-validation - point them at the same
clips if you want a learned model rather than tuned rules.